In [1]:
# PEG BOARD TEST - FULL BIO PIPELINE
import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
import math
import csv
import ctypes

In [2]:
# MediaPipe Setup (Hands + Pose)
mpHands = mp.solutions.hands
hands = mpHands.Hands(min_detection_confidence=0.6)

mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

mpDraw = mp.solutions.drawing_utils

DRAWING_SPEC_LANDMARK = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)
DRAWING_SPEC_CONNECTION = mpDraw.DrawingSpec(color=(0,0,0), thickness=2)

In [3]:
# Utility Functions

def safe_mean(arr, default=0.0):
    return sum(arr)/len(arr) if len(arr) > 0 else default

def safe_std(arr, default=0.0):
    return float(np.std(arr)) if len(arr) > 0 else default

In [4]:
def get_display_size(frame_width, frame_height, margin=100):
    try:
        user32 = ctypes.windll.user32
        screen_width = user32.GetSystemMetrics(0)
        screen_height = user32.GetSystemMetrics(1)
    except Exception:
        screen_width, screen_height = frame_width, frame_height

    max_width = max(screen_width - margin, 1)
    max_height = max(screen_height - margin, 1)
    scale = min(max_width / max(frame_width, 1), max_height / max(frame_height, 1))

    return max(1, int(frame_width * scale)), max(1, int(frame_height * scale))

In [5]:
# HAND STABILITY (TREMOR)

def hand_stability(x_coords, y_coords):
    return safe_std(x_coords) + safe_std(y_coords)

In [6]:
# MOVEMENT METRICS (UPDATED - SAFE & ALIGNED)
def compute_metrics(left_y, right_y, time_list):

    left_y = np.array(left_y)
    right_y = np.array(right_y)
    t = np.array(time_list)

    # -------------------------------
    # ALIGN ALL ARRAYS (CRITICAL FIX)
    # -------------------------------
    min_len = min(len(left_y), len(right_y), len(t))

    if min_len < 3:
        return 0.2, 0.0  # fallback safe values

    left_y = left_y[:min_len]
    right_y = right_y[:min_len]
    t = t[:min_len]

    # -------------------------------
    # RHYTHM (LEFT HAND)
    # -------------------------------
    peaks_l, _ = find_peaks(-left_y, distance=3)
    peaks_l = peaks_l[peaks_l < min_len]   # safety filter

    if len(peaks_l) > 1:
        intervals_l = np.diff(t[peaks_l])
    else:
        intervals_l = np.array([0.2])

    # -------------------------------
    # RHYTHM (RIGHT HAND)
    # -------------------------------
    peaks_r, _ = find_peaks(-right_y, distance=3)
    peaks_r = peaks_r[peaks_r < min_len]   # safety filter

    if len(peaks_r) > 1:
        intervals_r = np.diff(t[peaks_r])
    else:
        intervals_r = np.array([0.2])

    # Combine both hands
    all_intervals = np.concatenate([intervals_l, intervals_r])
    rhythm = safe_std(all_intervals, 0.2)

    # -------------------------------
    # BILATERAL SYNCHRONIZATION
    # -------------------------------
    sync_diff = safe_mean(np.abs(left_y - right_y))

    return rhythm, sync_diff

In [7]:
# PRIMARY SCORE (PEG COUNT)
def peg_count_score(total_pegs):
    if total_pegs >= 20:
        return 5
    elif total_pegs >= 16:
        return 4
    elif total_pegs >= 12:
        return 3
    elif total_pegs >= 8:
        return 2
    else:
        return 1

In [8]:
# BILATERAL ADJUSTMENT
def bilateral_adjustment(base_score, left_ratio):
    if 0.4 <= left_ratio <= 0.6:
        return base_score
    elif 0.35 <= left_ratio <= 0.65:
        return base_score - 0.5
    elif 0.3 <= left_ratio <= 0.7:
        return base_score - 1
    else:
        return min(base_score, 2)

In [9]:
# ERROR ADJUSTMENT
def error_adjustment(base_score, errors):
    if errors <= 1:
        return base_score
    elif errors <= 3:
        return base_score - 0.5
    elif errors <= 5:
        return base_score - 1
    else:
        return min(base_score, 2)

In [10]:
# QUALITY SCORE (BIO)
def quality_score(avg_stability, rhythm, sync_diff, fatigue):
    quality = 0
    if avg_stability < 0.02: quality += 2
    elif avg_stability < 0.05: quality += 1

    if rhythm < 0.05: quality += 2
    elif rhythm < 0.08: quality += 1

    if sync_diff < 0.02: quality += 2
    elif sync_diff < 0.05: quality += 1

    if fatigue < 0.02: quality += 1

    return quality

In [11]:
# FINAL SCORE
def final_score_cal(base_score, quality):
    if base_score >= 4 and quality >= 6:
        return 5
    elif base_score >= 3 and quality >= 4:
        return 4
    elif base_score >= 3:
        return 3
    elif base_score == 2:
        return 2
    else:
        return 1

In [12]:
#category
def category_cal(final_score):
    if final_score == 5:
        return "Excellent"
    elif final_score == 4:
        return "Above Average"
    elif final_score == 3:
        return "Average"
    elif final_score == 2:
        return "Below Average" 
    else:
        return "Poor"

In [17]:
# -------------------------------
# MAIN FUNCTION
# -------------------------------
def peg_board(ID="ID001", name="Test", path="0", save_csv=True):

    cap = cv2.VideoCapture(path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(3))
    h = int(cap.get(4))
    display_width, display_height = get_display_size(w, h)
    window_name = "Peg Board Processing"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, display_width, display_height)

    left_y, right_y, time_list = [], [], []
    left_x_all, left_y_all = [], []
    right_x_all, right_y_all = [], []

    left_count = 0
    right_count = 0
    errors = 0

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        hand_results = hands.process(rgb)
        pose_results = pose.process(rgb)

        if hand_results.multi_hand_landmarks:
            for hand_landmarks in hand_results.multi_hand_landmarks:
                mpDraw.draw_landmarks(frame, hand_landmarks, mpHands.HAND_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)

                # index fingertip
                x = hand_landmarks.landmark[8].x
                y = hand_landmarks.landmark[8].y

                # classify left/right based on x
                if x < 0.5:
                    left_x_all.append(x)
                    left_y_all.append(y)
                    left_y.append(y)
                    pt = (int(x * w), int(y * h))
                    cv2.circle(frame, pt, 6, (0, 255, 0), -1)

                    # peg placement approx (downward motion)
                    if y > 0.6:
                        left_count += 1

                else:
                    right_x_all.append(x)
                    right_y_all.append(y)
                    right_y.append(y)
                    pt = (int(x * w), int(y * h))
                    cv2.circle(frame, pt, 6, (0, 255, 0), -1)

                    if y > 0.6:
                        right_count += 1

        # cv2.line(frame, (0, int(0.6 * h)), (w, int(0.6 * h)), (255, 0, 0), 2)
        # cv2.line(frame, (int(0.5 * w), 0), (int(0.5 * w), h), (255, 0, 0), 2)
        cv2.putText(frame, f"Frame: {frame_idx}  Pegs: {left_count + right_count}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2,)
        cv2.imshow(window_name, frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

        if hand_results.multi_hand_landmarks:
            time_list.append(frame_idx / fps if fps else 0)
        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    total_pegs = left_count + right_count

    # =========================================================
    # BIOLOGICAL METRICS

    rhythm, sync_diff = compute_metrics(left_y, right_y, time_list)

    # -------- HAND STABILITY --------
    left_stability = hand_stability(left_x_all, left_y_all)
    right_stability = hand_stability(right_x_all, right_y_all)

    avg_stability = (left_stability + right_stability) / 2

    # -------- BILATERAL RATIO --------
    total = max(total_pegs, 1)
    left_ratio = left_count / total
    right_ratio = right_count / total

    # -------- FATIGUE --------
    mid = len(left_y) // 2
    fatigue = abs(safe_mean(left_y[:mid]) - safe_mean(left_y[mid:]))
    

    base_score = 0
    
    # PRIMARY SCORE (PEG COUNT)
    base_score = peg_count_score(total_pegs)

    # BILATERAL ADJUSTMENT
    base_score = bilateral_adjustment(base_score, left_ratio)

    # ERROR ADJUSTMENT
    base_score = error_adjustment(base_score, errors)

    # QUALITY SCORE (BIO)
    quality = quality_score(avg_stability, rhythm, sync_diff, fatigue)

    # FINAL SCORE
    # print("base_score:", base_score, type(base_score))
    # print("quality:", quality, type(quality))
    # print(base_score)
    final_score = final_score_cal(base_score, quality)

    #category
    category = category_cal(final_score)

    # =========================================================
    # OUTPUT
    print("------ PEG BOARD RESULT ------")
    print(f"Total Pegs: {total_pegs}")
    print(f"Left: {left_count}, Right: {right_count}")
    print(f"Balance Ratio: {left_ratio:.2f}/{right_ratio:.2f}")
    print(f"Stability: {avg_stability:.3f}")
    print(f"Rhythm: {rhythm:.3f}")
    print(f"Sync: {sync_diff:.3f}")
    print(f"Fatigue: {fatigue:.3f}")
    print(f"Final Score: {final_score}")
    print(f"Category: {category}")

    return final_score, category


In [18]:
path = "data/peg_board_1.mp4"
peg_board(ID="ID001", name="Test", path=path, save_csv=True)

------ PEG BOARD RESULT ------
Total Pegs: 3
Left: 0, Right: 3
Balance Ratio: 0.00/1.00
Stability: 0.095
Rhythm: 0.200
Sync: 0.000
Fatigue: 0.000
Final Score: 1
Category: Poor


(1, 'Poor')